# Chapter 7: Working with Text Data

## Overview
This notebook covers processing, vectorizing, and modeling textual data:
1.  **Bag-of-Words Representation**: Tokenization, vocabulary extraction, and sparse encoding via `CountVectorizer`.
2.  **Text Normalization**: Stop-word filtering, n-grams, and TF-IDF scaling (`TfidfVectorizer`).
3.  **Topic Modeling & Unsupervised Inspection**: Extracting latent document topics using Non-negative Matrix Factorization (NMF) and Latent Dirichlet Allocation (LDA).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline


# Matplotlib global settings
plt.rc("font", size=10)
plt.rc("axes", labelsize=11, titlesize=12)

## 1. Bag-of-Words (BoW) Representation

The **Bag-of-Words** approach converts unstructured text into numerical feature vectors:
1.  **Tokenization**: Splitting text documents into individual word tokens.
2.  **Vocabulary Building**: Collecting every unique word token across all documents.
3.  **Sparse Encoding**: Counting occurrence frequencies for each vocabulary word per document.

In [ ]:
# Corpus of sample documents
corpus = [
    "Machine learning algorithms extract patterns from text data.",
    "Text processing and natural language processing are core tasks.",
    "Supervised learning algorithms classify text and tabular data."
]

vect = CountVectorizer()
X_bow = vect.fit_transform(corpus)

print("Extracted Vocabulary Length:", len(vect.get_feature_names_out()))
print("Vocabulary mapping:\n", vect.vocabulary_)

# Convert sparse matrix output to dense DataFrame for structural inspection
df_bow = pd.DataFrame(X_bow.toarray(), columns=vect.get_feature_names_out())
print("\nBag-of-Words Sparse Frequency Table:")
display(df_bow)

## 2. Text Normalization: Stop Words, N-Grams, and TF-IDF

High-frequency grammatical words (e.g., "and", "the", "is") add feature dimension noise without providing domain semantics.

-   **Stop Words Filtering**: Dropping common language-specific grammatical words.
-   **N-Grams**: Capturing token context sequences ($1$-grams = unigrams, $2$-grams = bigrams, $3$-grams = trigrams).
-   **TF-IDF Scaling**: Weighing term frequency against inverse document frequency to downweight words that appear frequently across *all* documents:

$$\text{TF-IDF}(w, d) = \text{TF}(w, d) \cdot \left( \log \frac{1 + N}{1 + \text{DF}(w)} + 1 \right)$$

In [ ]:
# Sample movie reviews dataset
reviews = [
    "This movie was amazingly great and very inspiring!",
    "Terrible performance, awful plot, absolutely not recommended.",
    "Great direction and solid acting, really enjoyed it.",
    "Boring movie, bad script, and terrible execution."
]
labels = [1, 0, 1, 0]  # 1: Positive, 0: Negative

# Instantiate TF-IDF Vectorizer with English stop words and (1, 2)-grams
tfidf_vect = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
X_tfidf = tfidf_vect.fit_transform(reviews)

df_tfidf = pd.DataFrame(
    X_tfidf.toarray(), columns=tfidf_vect.get_feature_names_out()
)
print("TF-IDF Feature Matrix with (1, 2)-grams:")
display(df_tfidf.round(3))

# Train Logistic Regression classifier on TF-IDF features
clf = LogisticRegression().fit(X_tfidf, labels)
print(f"Model Training Accuracy: {clf.score(X_tfidf, labels)*100:.2f}%")

## 3. Topic Modeling: Unsupervised Text Inspection

Topic modeling algorithms extract latent semantic themes across document collections without target class labels:
-   **NMF (Non-negative Matrix Factorization)**: Decomposes term-document matrices into non-negative component matrices ($V \approx W \cdot H$). Yields distinct, interpretable topics.
-   **LDA (Latent Dirichlet Allocation)**: Probabilistic generative model assuming documents are mixtures over latent topics, and topics are mixtures over words.

In [ ]:
# Synthetic news/document corpus
tech_news_corpus = [
    "Neural networks and deep learning models require heavy GPU hardware computation.",
    "Cryptocurrency markets and blockchain technology rely on distributed consensus algorithms.",
    "Hardware accelerators optimize matrix multiplication for deep neural net architectures.",
    "Decentralized financial networks run on cryptographically secure smart contract ledgers.",
    "Convolutional layers and transformers represent modern deep learning approaches."
]

# Extract raw word counts
cv_topic = CountVectorizer(stop_words="english")
X_topics = cv_topic.fit_transform(tech_news_corpus)
feature_names = cv_topic.get_feature_names_out()

# Fit Non-negative Matrix Factorization (NMF) for 2 topics
nmf = NMF(n_components=2, random_state=42)
nmf.fit(X_topics)

print("--- NMF Extracted Topics (Top Words) ---")
for topic_idx, topic in enumerate(nmf.components_):
    top_word_indices = topic.argsort()[:-6:-1]
    top_words = [feature_names[i] for i in top_word_indices]
    print(f"Topic #{topic_idx + 1}: {', '.join(top_words)}")

# Fit Latent Dirichlet Allocation (LDA) for 2 topics
lda = LatentDirichletAllocation(n_components=2, random_state=42)
lda.fit(X_topics)

print("\n--- LDA Extracted Topics (Top Words) ---")
for topic_idx, topic in enumerate(lda.components_):
    top_word_indices = topic.argsort()[:-6:-1]
    top_words = [feature_names[i] for i in top_word_indices]
    print(f"Topic #{topic_idx + 1}: {', '.join(top_words)}")